# AnyText Document Generation Feasibility Study (Colab GPU)

**Mục tiêu**: Đánh giá thực nghiệm editor chuyên biệt văn bản **AnyText** (ICLR 2024 Spotlight) trên đúng 2 nguồn thay thế chữ/số (Attempt A & B) từ pilot SDXL đã đóng.
- **Checkpoint**: `damo/cv_anytext_text_generation_editing` (ModelScope v1.1.3, ~6.93 GiB FP16; tắt và loại bỏ translator để bảo toàn ngân sách 12 GiB).
- **Môi trường Python**: Thiết lập môi trường Python 3.10 riêng biệt (`/content/anytext_env`) khớp hoàn toàn `environment.yaml` upstream; không can thiệp Python hệ thống của Colab (Python 3.13) và không nâng packages lên latest.
- **Dữ liệu đầu vào**: Tải trực tiếp từ archive SDXL đã có (`document_feasibility_results.zip`, SHA-256 `deadd9205207...`). Không tải lại shard CORD parquet.
- **Bảo toàn hình học**: Native dimensions, context crop 512x512, strict composite inside target bbox, kiểm tra bitwise outside-mask equality.
- **Yêu cầu môi trường**: Google Colab GPU (Tesla T4 miễn phí, >= 15 GB disk, zero-token, zero-login).

### Cấu trúc và Thứ tự Thực thi Notebook (Tổng cộng 7 cells: 1 Markdown + 6 Code cells):
- **Cell 0 [Markdown]**: Tổng quan mục tiêu, thông số checkpoint, môi trường và hướng dẫn thực thi.
- **Cell 1 [Code]**: Hardware Preflight & Disk Inspection (Xác minh GPU VRAM >= 7 GB, Disk >= 12 GB).
- **Cell 2 [Code]**: Setup Dedicated Python 3.10 Runtime, Clone AnyText & Verify Dependencies (Cài đặt môi trường Python 3.10 cô lập, pip check 0 lỗi, smoke test modules & functional glyph/position guidance).
- **Cell 3 [Code]**: Staging Geometry Modules, Runner & Plan with Bit-Exact Verification (Staging 4 files với SHA-256 đối soát bit-exact).
- **Cell 4 [Code]**: Upload and Verify SDXL Feasibility Archive (Tải lên document_feasibility_results.zip, SHA-256 deadd9205207...).
- **Cell 5 [Code]**: Execute AnyText Feasibility Runner using Dedicated Python 3.10 Environment (Chạy python -u run_anytext_colab.py, stream log trực tiếp, bắt lỗi stage chi tiết, tạo failure receipt và partial results zip nếu lỗi).
- **Cell 6 [Code]**: Visual Inspection & Download Whitelisted Results Archive (Hiển thị contact sheet và tải về kết quả <= 100 MiB).

In [ ]:
# Cell 1: Hardware Preflight & Disk Inspection
import torch, shutil, os

assert torch.cuda.is_available(), 'ERROR: GPU runtime required! Please enable GPU via Runtime -> Change runtime type.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
disk_free_gb = round(shutil.disk_usage('/').free / (1024**3), 2)

print(f'[PASS] GPU Detected: {gpu_name} (VRAM: {vram_gb} GB)')
print(f'[PASS] Disk Free: {disk_free_gb} GB (Requires >= 15 GB for ModelScope download)')
assert vram_gb >= 7.0, f'VRAM {vram_gb} GB is insufficient for AnyText (requires >= 7 GB).'
assert disk_free_gb >= 12.0, f'Disk space {disk_free_gb} GB is insufficient for 6.93 GiB checkpoint.'

In [ ]:
# Cell 2: Setup Dedicated Python 3.10 Runtime, Clone AnyText & Verify Dependencies
import os, pathlib, shutil, subprocess, sys

# 1. Install system font
subprocess.run(['apt-get', 'update', '-qq'], check=False, stdout=subprocess.DEVNULL)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-dejavu-core'], check=False)

# 2. Clone AnyText official repository to absolute path /content/AnyText and checkout locked commit
anytext_repo_dir = pathlib.Path('/content/AnyText')
locked_commit = '3d4a260c67d92c0e27d6e18ed69990e19a40de6d'
if not (anytext_repo_dir / '.git').is_dir():
    print(f'[INFO] Cloning AnyText repository into {anytext_repo_dir}...')
    subprocess.run(['git', 'clone', 'https://github.com/tyxsspa/AnyText.git', str(anytext_repo_dir)], check=True)

subprocess.run(['git', 'checkout', locked_commit], cwd=str(anytext_repo_dir), check=True)
head_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(anytext_repo_dir)).decode('utf-8').strip()
assert head_commit == locked_commit, f'Git commit mismatch! Expected {locked_commit}, got {head_commit}'
os.chdir(str(anytext_repo_dir))
print(f'[PASS] AnyText codebase verified at absolute path {anytext_repo_dir} (commit: {head_commit})')

# 3. Create isolated Python 3.10 environment (/content/anytext_env)
# Host Python on Colab may be 3.13; we explicitly target Python 3.10
env_dir = pathlib.Path('/content/anytext_env')
py_bin = env_dir / 'bin' / 'python'
pip_bin = env_dir / 'bin' / 'pip'

if not py_bin.is_file():
    py310_sys = shutil.which('python3.10')
    if not py310_sys:
        print('[INFO] Installing python3.10 and python3.10-venv from deadsnakes PPA...')
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'software-properties-common'], check=False, stdout=subprocess.DEVNULL)
        subprocess.run(['add-apt-repository', '-y', 'ppa:deadsnakes/ppa'], check=False, stdout=subprocess.DEVNULL)
        subprocess.run(['apt-get', 'update', '-qq'], check=False, stdout=subprocess.DEVNULL)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'python3.10', 'python3.10-venv', 'python3.10-dev'], check=False)
        py310_sys = shutil.which('python3.10')

    if py310_sys:
        print(f'[INFO] Creating virtualenv using system {py310_sys}...')
        subprocess.run([py310_sys, '-m', 'venv', str(env_dir)], check=True)
    else:
        print('[INFO] System python3.10 not found. Bootstrapping standalone Python 3.10 via uv...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', str(env_dir), '--python', '3.10', '--seed'], check=True)

assert py_bin.is_file(), f'Dedicated Python 3.10 binary failed to create at {py_bin}'
py_ver = subprocess.check_output([str(py_bin), '--version']).decode('utf-8').strip()
print(f'[PASS] Dedicated Python environment verified: {py_ver} at {py_bin}')
assert '3.10' in py_ver, f'Expected Python 3.10 environment, but found {py_ver}! Refusing to proceed.'

# 4. Lock installer toolchain versions (avoid unverified latest upgrades)
print('[INFO] Locking installer versions (pip==23.3.2, setuptools==68.2.2, wheel==0.42.0)...')
subprocess.run([str(pip_bin), 'install', 'pip==23.3.2', 'setuptools==68.2.2', 'wheel==0.42.0'], check=True)

# 5. Install PyTorch 2.0.1 (CUDA 11.8) matching upstream environment.yaml
print('[INFO] Installing PyTorch 2.0.1 (CUDA 11.8) matching upstream environment.yaml...')
subprocess.run([
    str(pip_bin), 'install',
    'torch==2.0.1', 'torchvision==0.15.2',
    '--index-url', 'https://download.pytorch.org/whl/cu118'
], check=True)

# 6. Install locked AnyText inference dependencies from upstream environment.yaml & import graph
print('[INFO] Installing AnyText locked inference dependencies...')
deps = [
    'numpy==1.23.3',
    'Pillow==9.5.0',
    'torchmetrics==0.11.4',
    'pytorch-lightning==1.5.0',
    'omegaconf==2.2.3',
    'einops==0.4.1',
    'opencv-python==4.7.0.72',
    'transformers==4.30.2',
    'open_clip_torch==2.7.0',
    'modelscope==1.10.0',
    'addict==2.4.0',
    'yapf==0.32.0',
    'timm==0.6.7',
    'safetensors==0.4.0',
    'easydict==1.10',
    'scikit-image==0.20.0',
    'scipy==1.10.1',
    'six==1.16.0',
    'tqdm==4.66.1',
    'ujson==5.9.0'
]
subprocess.run([str(pip_bin), 'install'] + deps, check=True)

# 7. pip check verification
print('[INFO] Executing pip check on dedicated environment...')
chk_res = subprocess.run([str(pip_bin), 'check'], capture_output=True, text=True)
print(chk_res.stdout)
if chk_res.stderr:
    print(chk_res.stderr)
assert chk_res.returncode == 0, f'pip check failed with dependency conflict:\n{chk_res.stdout}\n{chk_res.stderr}'
print('[PASS] pip check passed with 0 broken requirements.')

# 8. Import Smoke Test and Functional Glyph/Position Guidance Test
print('[INFO] Executing import and functional smoke test in Python 3.10...')
smoke_code = (
    'import sys\n'
    'print(f"[SMOKE] Python interpreter: {sys.executable} ({sys.version})")\n'
    'assert "3.10" in sys.version, f"Wrong python version: {sys.version}"\n'
    '\n'
    'import torch, torchvision, numpy as np, PIL, pytorch_lightning as pl, torchmetrics, ujson\n'
    'print(f"[SMOKE] Package versions: torch={torch.__version__}, torchvision={torchvision.__version__}, numpy={np.__version__}, PIL={PIL.__version__}, pl={pl.__version__}, torchmetrics={torchmetrics.__version__}, ujson={ujson.__version__}")\n'
    'assert np.__version__ == "1.23.3", f"numpy version mismatch: {np.__version__}"\n'
    'assert PIL.__version__ == "9.5.0", f"PIL version mismatch: {PIL.__version__}"\n'
    'assert pl.__version__ == "1.5.0", f"pl version mismatch: {pl.__version__}"\n'
    'assert torchmetrics.__version__ == "0.11.4", f"torchmetrics version mismatch: {torchmetrics.__version__}"\n'
    'assert ujson.__version__ == "5.9.0", f"ujson version mismatch: {ujson.__version__}"\n'
    'assert torch.__version__.startswith("2.0.1"), f"torch version mismatch: {torch.__version__}"\n'
    'assert torchvision.__version__.startswith("0.15.2"), f"torchvision version mismatch: {torchvision.__version__}"\n'
    '\n'
    'assert torch.cuda.is_available(), "CUDA is not available in PyTorch! Check Colab GPU runtime."\n'
    'print(f"[SMOKE CUDA] Device: {torch.cuda.get_device_name(0)}, CUDA version: {torch.version.cuda}")\n'
    '\n'
    'import cv2, einops, omegaconf, transformers, open_clip, addict, yapf, timm, safetensors, easydict, skimage, six, tqdm, ujson, modelscope\n'
    'from modelscope.pipelines import pipeline\n'
    'import cldm.model, cldm.cldm, cldm.ddim_hacked, cldm.recognizer, t3_dataset, util, bert_tokenizer\n'
    'print("[PASS] Import smoke test: 100% of required inference modules and AnyText submodules imported cleanly!")\n'
    '\n'
    '# Functional Smoke Test: Generate glyph and position guidance with DejaVuSans for both target texts without loading checkpoint\n'
    'from PIL import ImageFont\n'
    'font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"\n'
    'font = ImageFont.truetype(font_path, size=60)\n'
    '\n'
    '# Target 1: "350,000"\n'
    't1 = "350,000"\n'
    'g1 = t3_dataset.draw_glyph(font, t1)\n'
    'assert g1.shape == (80, 512, 1), f"Unexpected glyph shape: {g1.shape}"\n'
    'assert g1.sum() > 0, "Target 1 glyph rasterization has 0 pixels!"\n'
    'poly1 = np.array([[100, 200], [300, 200], [300, 280], [100, 280]])\n'
    'gp1 = t3_dataset.draw_glyph2(font, t1, poly1, width=512, height=512)\n'
    'assert gp1.shape == (512, 512, 1), f"Unexpected pos shape: {gp1.shape}"\n'
    'assert gp1.sum() > 0, "Target 1 position guidance has 0 pixels!"\n'
    '\n'
    '# Target 2: "Magnum Gold Almond"\n'
    't2 = "Magnum Gold Almond"\n'
    'g2 = t3_dataset.draw_glyph(font, t2)\n'
    'assert g2.shape == (80, 512, 1), f"Unexpected glyph shape: {g2.shape}"\n'
    'assert g2.sum() > 0, "Target 2 glyph rasterization has 0 pixels!"\n'
    'poly2 = np.array([[50, 150], [450, 150], [450, 250], [50, 250]])\n'
    'gp2 = t3_dataset.draw_glyph2(font, t2, poly2, width=512, height=512)\n'
    'assert gp2.shape == (512, 512, 1), f"Unexpected pos shape: {gp2.shape}"\n'
    'assert gp2.sum() > 0, "Target 2 position guidance has 0 pixels!"\n'
    '\n'
    'print(f"[PASS] Functional smoke test passed: glyphs & position guidance generated for \'{t1}\' (glyph: {int(g1.sum())}, pos: {int(gp1.sum())}) and \'{t2}\' (glyph: {int(g2.sum())}, pos: {int(gp2.sum())}).")\n'
)
smoke_env = os.environ.copy()
smoke_env['PYTHONPATH'] = str(anytext_repo_dir)
smoke_res = subprocess.run([str(py_bin), '-c', smoke_code], capture_output=True, text=True, cwd=str(anytext_repo_dir), env=smoke_env)
print(smoke_res.stdout)
if smoke_res.stderr:
    print(smoke_res.stderr)
assert smoke_res.returncode == 0, f'Smoke test failed:\n{smoke_res.stderr}'

In [ ]:
# Cell 3: Staging Geometry Modules, Runner & Plan with Bit-Exact Verification
import os, pathlib, hashlib

repo_dir = pathlib.Path('/content/AnyText')
os.chdir(str(repo_dir))

(repo_dir / 'ml/evaluation').mkdir(parents=True, exist_ok=True)
(repo_dir / 'research/evidence/document_pilot').mkdir(parents=True, exist_ok=True)

p_plan = repo_dir / 'research/evidence/document_pilot/anytext_feasibility_plan.json'
p_plan.write_bytes('{\n  "plan_id": "anytext_feasibility_plan_v1.0.0",\n  "protocol_version": "1.3.0",\n  "created_at": "2026-10-13T20:00:00+07:00",\n  "official_project_title": "Nghiên cứu và xây dựng công cụ phát hiện ảnh và tài liệu giả mạo sinh bởi AI.",\n  "objective": "Evaluate specialized text editor AnyText on two replacement sources from the closed SDXL pilot to determine feasibility of generating forensically relevant text-edited document samples without full dataset downloads or training.",\n  "model_specification": {\n    "model_name": "AnyText: Multilingual Visual Text Generation And Editing",\n    "paper_url": "https://arxiv.org/abs/2311.03054",\n    "official_repo_url": "https://github.com/tyxsspa/AnyText",\n    "code_commit_sha": "3d4a260c67d92c0e27d6e18ed69990e19a40de6d",\n    "modelscope_repo_id": "damo/cv_anytext_text_generation_editing",\n    "model_revision": "v1.1.3",\n    "checkpoint_file": "anytext_v1.1.ckpt",\n    "checkpoint_sha256": "09c88d145aa08f067641dc1d10c8890855939fa756d4cdc991e781a5f3918f59",\n    "checkpoint_size_bytes": 5730281759,\n    "clip_subfolder": "clip-vit-large-patch14",\n    "clip_weights_size_bytes": 1710671599,\n    "total_required_weights_bytes": 7440953358,\n    "total_required_weights_gib": 6.9299,\n    "translator_subfolder_excluded": "nlp_csanmt_translation_zh2en (7,881,046,036 bytes / 7.34 GiB excluded to respect 12 GiB ceiling)",\n    "translator_policy": "DISABLED (use_translator=False, --no_translator)",\n    "dtype": "fp16",\n    "licenses": {\n      "code_license": "Apache License 2.0",\n      "modelscope_model_license": "Apache License 2.0",\n      "font_license": "Bitstream Vera / Free Software License (SIL OFL compatible)"\n    },\n    "font_specification": {\n      "font_name": "DejaVuSans.ttf",\n      "font_package": "fonts-dejavu-core",\n      "font_path_default_linux": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n      "font_path_fallback_linux": "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",\n      "font_size_pt": 60,\n      "font_source": "Debian/Ubuntu system package (open source, verified free distribution)"\n    }\n  },\n  "resource_budget": {\n    "max_colab_download_gib": 12.0,\n    "planned_colab_download_gib": 6.93,\n    "max_local_transfer_mib": 100.0,\n    "max_gpu_execution_minutes": 60.0,\n    "authentication_required": false\n  },\n  "input_source_specification": {\n    "archive_path": "data/research/local-artifacts/document-feasibility/document_feasibility_results.zip",\n    "archive_sha256": "deadd9205207eaaa296a8d9eeb493ee48eeea76ee7afd210684187e4026ed684",\n    "archive_size_bytes": 10592936,\n    "extraction_policy": "SAFE_EXTRACT_NO_PARENT_DIR_NO_ABSOLUTE_PATH",\n    "authentic_controls": {\n      "attempt_A": {\n        "source_path_in_zip": "attempt_0_replacement_price/authentic_control.png",\n        "sha256": "b335696aaeb1ba1a607e87cd7acf76798dbc65a188b2f38e347473a762aba087"\n      },\n      "attempt_B": {\n        "source_path_in_zip": "attempt_1_replacement_item_name/authentic_control.png",\n        "sha256": "77d047b9791ff4667bf85b0b86448daf154af15660686cf72047d9db407b5ed0"\n      }\n    },\n    "sdxl_pilot_outputs": {\n      "attempt_A": "attempt_0_replacement_price/final_edited_document.png",\n      "attempt_B": "attempt_1_replacement_item_name/final_edited_document.png"\n    }\n  },\n  "geometry_contract": {\n    "native_document_dimensions_preserved": true,\n    "intermediate_canvas_resolution": [\n      512,\n      512\n    ],\n    "context_margin_ratio": 0.5,\n    "interpolation_image": "BICUBIC",\n    "interpolation_mask": "NEAREST (strictly binary {0, 255})",\n    "outside_mask_equality_contract": "np.array_equal == True, max_outside_diff == 0",\n    "resolution_difference_vs_sdxl": "AnyText uses SD1.5 latent resolution (512x512 canvas) whereas SDXL pilot used 1024x1024. This is a known editor/configuration difference, not an isolated architecture study.",\n    "post_generation_policy": "Raw model output and final composited document are saved separately. No post-generation typesetting or text stamping is performed."\n  },\n  "attempts": [\n    {\n      "attempt_id": "attempt_A_replacement_price",\n      "source_row_index": 22,\n      "source_image_id": 22,\n      "operation": "replacement",\n      "original_text": "280,000",\n      "target_text": "350,000",\n      "target_bbox": [\n        442,\n        406,\n        468,\n        488\n      ],\n      "native_dimensions": [\n        576,\n        864\n      ],\n      "authentic_sha256": "b335696aaeb1ba1a607e87cd7acf76798dbc65a188b2f38e347473a762aba087",\n      "seed": 20261012,\n      "prompt": "A receipt bill with price \\"350,000\\"",\n      "a_prompt": "best quality, extremely detailed, clear text edges, neat writing, no watermarks",\n      "n_prompt": "low-res, bad anatomy, cropped, worst quality, low quality, watermark, unreadable text, messy words, distorted text",\n      "sampling_parameters": {\n        "mode": "text-editing",\n        "ddim_steps": 20,\n        "cfg_scale": 9.0,\n        "strength": 1.0,\n        "eta": 0.0,\n        "canvas_width": 512,\n        "canvas_height": 512,\n        "context_margin_ratio": 0.5\n      }\n    },\n    {\n      "attempt_id": "attempt_B_replacement_item_name",\n      "source_row_index": 61,\n      "source_image_id": 61,\n      "operation": "replacement",\n      "original_text": "Populaire Chocolate",\n      "target_text": "Magnum Gold Almond",\n      "target_bbox": [\n        1085,\n        128,\n        1156,\n        749\n      ],\n      "native_dimensions": [\n        1356,\n        2315\n      ],\n      "authentic_sha256": "77d047b9791ff4667bf85b0b86448daf154af15660686cf72047d9db407b5ed0",\n      "seed": 20261013,\n      "prompt": "A receipt bill menu with item name \\"Magnum Gold Almond\\"",\n      "a_prompt": "best quality, extremely detailed, clear text edges, neat writing, no watermarks",\n      "n_prompt": "low-res, bad anatomy, cropped, worst quality, low quality, watermark, unreadable text, messy words, distorted text",\n      "sampling_parameters": {\n        "mode": "text-editing",\n        "ddim_steps": 20,\n        "cfg_scale": 9.0,\n        "strength": 1.0,\n        "eta": 0.0,\n        "canvas_width": 512,\n        "canvas_height": 512,\n        "context_margin_ratio": 0.5\n      }\n    }\n  ]\n}\n'.encode('utf-8'))
assert hashlib.sha256(p_plan.read_bytes()).hexdigest() == '88f7e32873ff4c4e96dc60aae33a50aee677a550291e3316a027e1584e69b3ad', 'Plan SHA mismatch after staging!'

p_geom = repo_dir / 'ml/evaluation/document_feasibility_geometry.py'
p_geom.write_bytes('"""Document feasibility geometry transformations and composite contract.\n\nImplements aspect-ratio preserving context cropping, canvas letterboxing,\nbinary mask transforms, inverse mapping, and strict bitwise outside-mask\nequality verification for AI document manipulation experiments.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom typing import Any, Tuple, Dict, List\nimport numpy as np\nfrom PIL import Image\n\n\ndef extract_roi_and_build_canvas(\n    image: Image.Image,\n    target_bbox: List[int] | Tuple[int, int, int, int],\n    canvas_size: int = 1024,\n    context_margin_ratio: float = 0.5,\n) -> Dict[str, Any]:\n    """Extract context crop around target bounding box and letterbox onto square canvas.\n\n    Args:\n        image: PIL Image in RGB format (native dimensions).\n        target_bbox: [ymin, xmin, ymax, xmax] in original pixel coordinates.\n        canvas_size: Square canvas resolution for diffusion model (e.g. 1024).\n        context_margin_ratio: Margin to expand around target bbox (e.g. 0.5 = 50%).\n\n    Returns:\n        Dict containing:\n            canvas_image: PIL Image (RGB, canvas_size x canvas_size)\n            canvas_mask: PIL Image (L, canvas_size x canvas_size, strictly {0, 255})\n            context_crop: PIL Image (RGB, crop_w x crop_h)\n            geometry_receipt: Dict detailing all transformations for inverse mapping.\n    """\n    if image.mode != "RGB":\n        image = image.convert("RGB")\n\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n\n    # Validate bounding box strictly: silent clamping is forbidden\n    if ymin < 0 or xmin < 0 or ymax > orig_h or xmax > orig_w:\n        raise ValueError(\n            f"Bounding box {target_bbox} is out of bounds for image dimensions ({orig_w}, {orig_h}). "\n            f"Silent clamping is forbidden by protocol."\n        )\n    if ymin >= ymax or xmin >= xmax:\n        raise ValueError(\n            f"Degenerate bounding box {target_bbox}: ymin ({ymin}) >= ymax ({ymax}) "\n            f"or xmin ({xmin}) >= xmax ({xmax})."\n        )\n\n    bbox_w = xmax - xmin\n    bbox_h = ymax - ymin\n\n    # Expand margin\n    margin_x = int(round(bbox_w * context_margin_ratio))\n    margin_y = int(round(bbox_h * context_margin_ratio))\n\n    crop_ymin = max(0, ymin - margin_y)\n    crop_ymax = min(orig_h, ymax + margin_y)\n    crop_xmin = max(0, xmin - margin_x)\n    crop_xmax = min(orig_w, xmax + margin_x)\n\n    crop_w = crop_xmax - crop_xmin\n    crop_h = crop_ymax - crop_ymin\n\n    # Crop image\n    context_crop = image.crop((crop_xmin, crop_ymin, crop_xmax, crop_ymax))\n\n    # Build mask in crop coordinate space\n    # Target bbox relative to crop origin\n    target_rel_ymin = ymin - crop_ymin\n    target_rel_ymax = ymax - crop_ymin\n    target_rel_xmin = xmin - crop_xmin\n    target_rel_xmax = xmax - crop_xmin\n\n    mask_crop_arr = np.zeros((crop_h, crop_w), dtype=np.uint8)\n    mask_crop_arr[target_rel_ymin:target_rel_ymax, target_rel_xmin:target_rel_xmax] = 255\n    mask_crop = Image.fromarray(mask_crop_arr, mode="L")\n\n    # Aspect-ratio preserving scaling\n    scale = min(canvas_size / crop_w, canvas_size / crop_h)\n    scaled_w = max(1, min(canvas_size, int(round(crop_w * scale))))\n    scaled_h = max(1, min(canvas_size, int(round(crop_h * scale))))\n\n    scaled_image = context_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.BICUBIC)\n    # Mask resize MUST use NEAREST neighbor to avoid intermediate gray values\n    scaled_mask = mask_crop.resize((scaled_w, scaled_h), resample=Image.Resampling.NEAREST)\n\n    # Re-binarize to ensure exact {0, 255}\n    scaled_mask_arr = (np.array(scaled_mask) > 127).astype(np.uint8) * 255\n    scaled_mask = Image.fromarray(scaled_mask_arr, mode="L")\n\n    # Letterbox onto square canvas\n    pad_x = (canvas_size - scaled_w) // 2\n    pad_y = (canvas_size - scaled_h) // 2\n\n    canvas_image = Image.new("RGB", (canvas_size, canvas_size), color=(128, 128, 128))\n    canvas_image.paste(scaled_image, (pad_x, pad_y))\n\n    canvas_mask = Image.new("L", (canvas_size, canvas_size), color=0)\n    canvas_mask.paste(scaled_mask, (pad_x, pad_y))\n\n    geometry_receipt = {\n        "original_size": [orig_w, orig_h],\n        "target_bbox": [ymin, xmin, ymax, xmax],\n        "crop_bbox": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n        "crop_size": [crop_w, crop_h],\n        "scale": float(scale),\n        "scaled_size": [scaled_w, scaled_h],\n        "padding": [pad_y, pad_x],\n        "canvas_size": canvas_size,\n        "context_margin_ratio": float(context_margin_ratio),\n    }\n\n    return {\n        "canvas_image": canvas_image,\n        "canvas_mask": canvas_mask,\n        "context_crop": context_crop,\n        "geometry_receipt": geometry_receipt,\n    }\n\n\ndef composite_inpainted_patch(\n    original_image: Image.Image,\n    model_output_canvas: Image.Image,\n    geometry_receipt: Dict[str, Any],\n) -> Dict[str, Any]:\n    """Inverse map model canvas output back to original coordinates and composite strictly inside mask.\n\n    Guarantees that pixels outside the intervention mask remain 100% bitwise identical\n    to the authentic source image.\n\n    Args:\n        original_image: Authentic PIL Image (native dimensions).\n        model_output_canvas: Raw output from inpainting model (canvas_size x canvas_size).\n        geometry_receipt: Transformation parameters from extract_roi_and_build_canvas.\n\n    Returns:\n        Dict containing:\n            final_edited_image: PIL Image (RGB, native dimensions)\n            final_intervention_mask: PIL Image (L, native dimensions, strictly {0, 255})\n            composite_receipt: Summary metadata\n    """\n    if original_image.mode != "RGB":\n        original_image = original_image.convert("RGB")\n    if model_output_canvas.mode != "RGB":\n        model_output_canvas = model_output_canvas.convert("RGB")\n\n    orig_w, orig_h = geometry_receipt["original_size"]\n    crop_ymin, crop_xmin, crop_ymax, crop_xmax = geometry_receipt["crop_bbox"]\n    crop_w, crop_h = geometry_receipt["crop_size"]\n    scaled_w, scaled_h = geometry_receipt["scaled_size"]\n    pad_y, pad_x = geometry_receipt["padding"]\n    target_ymin, target_xmin, target_ymax, target_xmax = geometry_receipt["target_bbox"]\n\n    # 1. Unpad canvas\n    unpadded_patch = model_output_canvas.crop((pad_x, pad_y, pad_x + scaled_w, pad_y + scaled_h))\n\n    # 2. Resize unpadded patch back to crop size\n    restored_patch = unpadded_patch.resize((crop_w, crop_h), resample=Image.Resampling.BICUBIC)\n    restored_patch_arr = np.array(restored_patch)\n\n    # 3. Create full-frame candidate edited image\n    orig_arr = np.array(original_image)\n    candidate_full_arr = orig_arr.copy()\n    candidate_full_arr[crop_ymin:crop_ymax, crop_xmin:crop_xmax] = restored_patch_arr\n\n    # 4. Create final intervention mask in original coordinates\n    final_mask_arr = np.zeros((orig_h, orig_w), dtype=np.uint8)\n    final_mask_arr[target_ymin:target_ymax, target_xmin:target_xmax] = 255\n\n    # 5. Composite STRICTLY inside intervention mask:\n    # pixels where mask == 255 take candidate_full_arr, everywhere else takes orig_arr bit-for-bit\n    final_edited_arr = np.where(final_mask_arr[:, :, None] == 255, candidate_full_arr, orig_arr)\n\n    final_edited_image = Image.fromarray(final_edited_arr, mode="RGB")\n    final_intervention_mask = Image.fromarray(final_mask_arr, mode="L")\n\n    composite_receipt = {\n        "status": "COMPOSITED",\n        "original_size": [orig_w, orig_h],\n        "crop_bbox": [crop_ymin, crop_xmin, crop_ymax, crop_xmax],\n        "target_bbox": [target_ymin, target_xmin, target_ymax, target_xmax],\n        "unpadded_size": [scaled_w, scaled_h],\n        "restored_crop_size": [crop_w, crop_h],\n    }\n\n    return {\n        "final_edited_image": final_edited_image,\n        "final_intervention_mask": final_intervention_mask,\n        "composite_receipt": composite_receipt,\n    }\n\n\ndef verify_outside_equality(\n    authentic_image: Image.Image | np.ndarray,\n    edited_image: Image.Image | np.ndarray,\n    intervention_mask: Image.Image | np.ndarray,\n) -> Tuple[bool, int]:\n    """Verify that all pixels outside the intervention mask are strictly bitwise identical.\n\n    Returns:\n        (is_equal, max_absolute_diff)\n    """\n    auth_arr = np.array(authentic_image, dtype=np.uint8)\n    edit_arr = np.array(edited_image, dtype=np.uint8)\n    mask_arr = np.array(intervention_mask, dtype=np.uint8)\n\n    if auth_arr.shape[:2] != mask_arr.shape[:2] or edit_arr.shape[:2] != mask_arr.shape[:2]:\n        raise ValueError("Image dimensions and mask dimensions do not match.")\n\n    outside_indices = mask_arr == 0\n    if not np.any(outside_indices):\n        # Entire image is masked (should not happen in document pilot)\n        return True, 0\n\n    auth_outside = auth_arr[outside_indices]\n    edit_outside = edit_arr[outside_indices]\n\n    is_equal = bool(np.array_equal(auth_outside, edit_outside))\n    if is_equal:\n        return True, 0\n\n    diff = np.abs(auth_outside.astype(np.int32) - edit_outside.astype(np.int32))\n    max_diff = int(np.max(diff))\n    return False, max_diff\n\n\ndef compute_pixel_change_statistics(\n    authentic_image: Image.Image | np.ndarray,\n    edited_image: Image.Image | np.ndarray,\n    intervention_mask: Image.Image | np.ndarray,\n) -> Dict[str, Any]:\n    """Compute detailed pixel difference statistics inside and outside the intervention mask."""\n    auth_arr = np.array(authentic_image, dtype=np.int32)\n    edit_arr = np.array(edited_image, dtype=np.int32)\n    mask_arr = np.array(intervention_mask, dtype=np.uint8)\n\n    inside_mask = mask_arr == 255\n    outside_mask = mask_arr == 0\n\n    inside_pixels_count = int(np.sum(inside_mask))\n    outside_pixels_count = int(np.sum(outside_mask))\n\n    # Inside differences\n    if inside_pixels_count > 0:\n        inside_diff = np.abs(auth_arr[inside_mask] - edit_arr[inside_mask])\n        inside_changed = np.any(inside_diff > 0, axis=-1)\n        changed_pixels_count = int(np.sum(inside_changed))\n        changed_pixel_ratio = float(changed_pixels_count / inside_pixels_count)\n        inside_mean_l1 = float(np.mean(inside_diff))\n        inside_max_l1 = int(np.max(inside_diff))\n    else:\n        changed_pixels_count = 0\n        changed_pixel_ratio = 0.0\n        inside_mean_l1 = 0.0\n        inside_max_l1 = 0\n\n    # Outside differences\n    if outside_pixels_count > 0:\n        outside_diff = np.abs(auth_arr[outside_mask] - edit_arr[outside_mask])\n        outside_changed = np.any(outside_diff > 0, axis=-1)\n        outside_changed_count = int(np.sum(outside_changed))\n        outside_mean_l1 = float(np.mean(outside_diff))\n        outside_max_l1 = int(np.max(outside_diff))\n    else:\n        outside_changed_count = 0\n        outside_mean_l1 = 0.0\n        outside_max_l1 = 0\n\n    return {\n        "inside_mask_pixel_count": inside_pixels_count,\n        "changed_pixels_count": changed_pixels_count,\n        "changed_pixel_ratio": changed_pixel_ratio,\n        "inside_mean_l1": inside_mean_l1,\n        "inside_max_l1": inside_max_l1,\n        "outside_mask_pixel_count": outside_pixels_count,\n        "outside_changed_count": outside_changed_count,\n        "outside_mean_l1": outside_mean_l1,\n        "outside_max_l1": outside_max_l1,\n        "is_outside_bit_identical": (outside_changed_count == 0),\n    }\n'.encode('utf-8'))
assert hashlib.sha256(p_geom.read_bytes()).hexdigest() == '566d10670570a0f89f2dbde3dd56092866b1b632fe5f7a109c879399ac0c4c73', 'Geometry SHA mismatch after staging!'

p_adapt = repo_dir / 'ml/evaluation/anytext_document_adapter.py'
p_adapt.write_bytes('"""AnyText Document Adapter and Geometry Contract.\n\nProvides preprocessing, position mask conditioning, inverse mapping,\nand strict outside-mask compositing for AnyText text-editing on document images.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any, Dict, List, Tuple\nimport numpy as np\nfrom PIL import Image\n\nfrom ml.evaluation.document_feasibility_geometry import (\n    extract_roi_and_build_canvas,\n    composite_inpainted_patch,\n    verify_outside_equality,\n    compute_pixel_change_statistics,\n)\n\n\ndef prepare_anytext_editing_inputs(\n    image: Image.Image,\n    target_bbox: List[int] | Tuple[int, int, int, int],\n    canvas_size: int = 512,\n    context_margin_ratio: float = 0.5,\n) -> Dict[str, Any]:\n    """Prepares AnyText ori_image and draw_pos inputs from native document and target bbox.\n\n    Args:\n        image: Original PIL Image in RGB format (native dimensions).\n        target_bbox: [ymin, xmin, ymax, xmax] in native coordinates.\n        canvas_size: Square canvas resolution for AnyText (512x512).\n        context_margin_ratio: Margin expansion around target bbox (default: 0.5).\n\n    Returns:\n        Dict with:\n            ori_image_canvas: PIL Image (RGB, 512x512) for AnyText ori_image.\n            draw_pos_canvas: PIL Image (L, 512x512, {0, 255}) for AnyText draw_pos.\n            geometry_receipt: Dict tracking transformations for inverse mapping.\n            native_mask: PIL Image (L, orig_w x orig_h, {0, 255}).\n    """\n    if image.mode != "RGB":\n        image = image.convert("RGB")\n\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n    if ymin < 0 or xmin < 0 or ymax > orig_h or xmax > orig_w or ymin >= ymax or xmin >= xmax:\n        raise ValueError(f"Invalid bounding box {target_bbox} for image size ({orig_w}, {orig_h})")\n\n    # Use verified geometry builder with 512x512 canvas size\n    roi_data = extract_roi_and_build_canvas(\n        image=image,\n        target_bbox=target_bbox,\n        canvas_size=canvas_size,\n        context_margin_ratio=context_margin_ratio,\n    )\n\n    orig_w, orig_h = image.size\n    ymin, xmin, ymax, xmax = [int(v) for v in target_bbox]\n\n    # Native intervention mask\n    native_mask_arr = np.zeros((orig_h, orig_w), dtype=np.uint8)\n    native_mask_arr[ymin:ymax, xmin:xmax] = 255\n    native_mask = Image.fromarray(native_mask_arr, mode="L")\n\n    return {\n        "ori_image_canvas": roi_data["canvas_image"],\n        "draw_pos_canvas": roi_data["canvas_mask"],\n        "context_crop": roi_data["context_crop"],\n        "geometry_receipt": roi_data["geometry_receipt"],\n        "native_mask": native_mask,\n    }\n\n\ndef composite_anytext_editing_output(\n    original_image: Image.Image,\n    raw_anytext_output: Image.Image | np.ndarray,\n    geometry_receipt: Dict[str, Any],\n    native_mask: Image.Image,\n) -> Dict[str, Any]:\n    """Composites raw AnyText canvas back into native document strictly inside mask.\n\n    Args:\n        original_image: Original PIL Image in RGB format.\n        raw_anytext_output: 512x512 RGB raw output from AnyText (PIL or numpy array).\n        geometry_receipt: Receipt from prepare_anytext_editing_inputs.\n        native_mask: Native binary mask (L mode, {0, 255}).\n\n    Returns:\n        Dict with:\n            final_edited_document: PIL Image in RGB format (native dimensions).\n            is_outside_equal: bool (True if 100% bitwise identical outside mask).\n            max_outside_diff: int (must be 0).\n            raw_model_canvas: PIL Image (512x512).\n    """\n    if original_image.mode != "RGB":\n        original_image = original_image.convert("RGB")\n\n    if isinstance(raw_anytext_output, np.ndarray):\n        raw_canvas = Image.fromarray(raw_anytext_output.astype(np.uint8), mode="RGB")\n    else:\n        raw_canvas = raw_anytext_output.convert("RGB")\n\n    # Composite using verified contract\n    comp_data = composite_inpainted_patch(\n        original_image=original_image,\n        model_output_canvas=raw_canvas,\n        geometry_receipt=geometry_receipt,\n    )\n    final_img = comp_data["final_edited_image"]\n    final_mask = comp_data["final_intervention_mask"]\n\n    # Strict outside-mask equality check\n    is_outside_equal, max_outside_diff = verify_outside_equality(\n        authentic_image=original_image,\n        edited_image=final_img,\n        intervention_mask=final_mask,\n    )\n\n    stats = compute_pixel_change_statistics(\n        authentic_image=original_image,\n        edited_image=final_img,\n        intervention_mask=final_mask,\n    )\n\n    return {\n        "final_edited_document": final_img,\n        "final_composited_document": final_img,\n        "final_intervention_mask": final_mask,\n        "is_outside_equal": is_outside_equal,\n        "max_outside_diff": max_outside_diff,\n        "raw_model_canvas": raw_canvas,\n        "composite_receipt": comp_data["composite_receipt"],\n        "statistics": stats,\n        "verification": {\n            "outside_equal": is_outside_equal,\n            "max_outside_diff": max_outside_diff,\n            "modified_pixel_count": stats["changed_pixels_count"],\n        },\n    }\n'.encode('utf-8'))
assert hashlib.sha256(p_adapt.read_bytes()).hexdigest() == 'd30342c40997f0a15c9a1a489232ad35399ba354761bf71cf5354ec46334a0c5', 'Adapter SHA mismatch after staging!'

p_runner = repo_dir / 'run_anytext_colab.py'
p_runner.write_bytes('#!/usr/bin/env python3\n"""AnyText Document Generation Feasibility Colab Runner.\n\nRuns AnyText text-editing on two locked replacement sources (Attempt A & B)\non Google Colab GPU without Hugging Face token or paid dependencies.\n\nInputs are extracted directly from the verified SDXL pilot archive:\n`data/research/local-artifacts/document-feasibility/document_feasibility_results.zip`\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport atexit\nimport csv\nimport datetime\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\n\ndef compute_sha256(path: Path) -> str:\n    """Computes SHA-256 hash of a file."""\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        while chunk := f.read(1024 * 1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef compute_bytes_sha256(data: bytes) -> str:\n    """Computes SHA-256 hash of raw bytes."""\n    return hashlib.sha256(data).hexdigest()\n\n\ndef check_gpu_and_resources(min_vram_gb: float = 7.0) -> Dict[str, Any]:\n    """Inspects CUDA device, VRAM, and disk storage."""\n    res: Dict[str, Any] = {\n        "cuda_available": False,\n        "device_name": "None",\n        "vram_total_gb": 0.0,\n        "vram_allocated_gb": 0.0,\n        "disk_free_gb": 0.0,\n    }\n    disk = shutil.disk_usage("/")\n    res["disk_free_gb"] = round(disk.free / (1024**3), 2)\n\n    try:\n        import torch\n        if torch.cuda.is_available():\n            res["cuda_available"] = True\n            res["device_name"] = torch.cuda.get_device_name(0)\n            res["vram_total_gb"] = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)\n            res["vram_allocated_gb"] = round(torch.cuda.memory_allocated(0) / (1024**3), 2)\n    except Exception as e:\n        res["error"] = str(e)\n    return res\n\n\ndef resolve_font_path(custom_path: Optional[str] = None) -> Tuple[str, str]:\n    """Finds an available open-source TrueType font for glyph rendering and returns (path, sha256)."""\n    candidates = [\n        custom_path,\n        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",\n        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",\n        "C:\\\\Windows\\\\Fonts\\\\arial.ttf",\n        "font/Arial_Unicode.ttf",\n    ]\n    for c in candidates:\n        if c and Path(c).is_file():\n            p = Path(c).resolve()\n            return str(p), compute_sha256(p)\n\n    # Fallback to apt install if on Debian/Ubuntu\n    try:\n        subprocess.run(["apt-get", "update", "-qq"], check=False)\n        subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-dejavu-core"], check=False)\n        target = Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")\n        if target.is_file():\n            return str(target), compute_sha256(target)\n    except Exception:\n        pass\n\n    raise FileNotFoundError("Could not find a valid TTF font for AnyText glyph rendering.")\n\n\ndef safe_extract_archive_inputs(\n    archive_path: Path,\n    target_extract_dir: Path,\n    plan: Dict[str, Any],\n) -> Dict[str, Dict[str, Any]]:\n    """Extracts authentic controls and SDXL outputs from document_feasibility_results.zip.\n\n    Verifies archive SHA-256 and byte-exact authentic control hashes.\n    Rejects directory traversal and absolute paths.\n    """\n    input_spec = plan.get("input_source_specification", {})\n    expected_archive_sha = input_spec.get("archive_sha256")\n\n    observed_archive_sha = compute_sha256(archive_path)\n    if expected_archive_sha and observed_archive_sha != expected_archive_sha:\n        raise ValueError(\n            f"Input archive SHA-256 mismatch! Expected {expected_archive_sha}, got {observed_archive_sha}"\n        )\n    print(f"[PASS] Archive SHA-256 verified: {observed_archive_sha[:16]}...")\n\n    target_extract_dir.mkdir(parents=True, exist_ok=True)\n    resolved_inputs: Dict[str, Dict[str, Any]] = {}\n\n    with zipfile.ZipFile(archive_path, "r") as zf:\n        namelist = set(zf.namelist())\n\n        # Path traversal security check\n        for member_name in namelist:\n            if member_name.startswith("/") or member_name.startswith("\\\\") or ".." in Path(member_name).parts:\n                raise SecurityError(f"Malicious member path detected in archive: {member_name}")\n\n        # Check manifest inside zip\n        manifest_data = {}\n        if "feasibility_manifest.json" in namelist:\n            manifest_data = json.loads(zf.read("feasibility_manifest.json").decode("utf-8"))\n\n        attempts = plan["attempts"]\n        for att in attempts:\n            att_id = att["attempt_id"]\n            row_idx = att.get("source_row_index")\n            expected_auth_sha = att["authentic_sha256"]\n\n            # Match attempt in archive\n            target_auth_member = None\n            target_sdxl_member = None\n\n            # 1. Try finding via manifest\n            for summ in manifest_data.get("attempts_summary", []):\n                if summ.get("source_image_id") == row_idx or summ.get("target_text") == att["target_text"]:\n                    m_auth_sha = summ.get("file_hashes", {}).get("authentic_control_sha256")\n                    if m_auth_sha == expected_auth_sha:\n                        folder = summ["attempt_id"]\n                        target_auth_member = f"{folder}/authentic_control.png"\n                        target_sdxl_member = f"{folder}/final_edited_document.png"\n                        break\n\n            # 2. Fallback to direct path conventions\n            if not target_auth_member:\n                if row_idx == 22:\n                    target_auth_member = "attempt_0_replacement_price/authentic_control.png"\n                    target_sdxl_member = "attempt_0_replacement_price/final_edited_document.png"\n                elif row_idx == 61:\n                    target_auth_member = "attempt_1_replacement_item_name/authentic_control.png"\n                    target_sdxl_member = "attempt_1_replacement_item_name/final_edited_document.png"\n\n            if target_auth_member not in namelist:\n                raise FileNotFoundError(f"Missing authentic member \'{target_auth_member}\' in archive for {att_id}")\n\n            # Extract raw authentic bytes directly without re-encoding\n            raw_auth_bytes = zf.read(target_auth_member)\n            obs_auth_sha = compute_bytes_sha256(raw_auth_bytes)\n            if obs_auth_sha != expected_auth_sha:\n                raise ValueError(\n                    f"Authentic image SHA-256 mismatch for {att_id}! Expected {expected_auth_sha}, got {obs_auth_sha}"\n                )\n\n            dest_att_dir = target_extract_dir / att_id\n            dest_att_dir.mkdir(parents=True, exist_ok=True)\n            dest_auth_path = dest_att_dir / "authentic_control.png"\n            dest_auth_path.write_bytes(raw_auth_bytes)\n\n            dest_sdxl_path = None\n            if target_sdxl_member and target_sdxl_member in namelist:\n                dest_sdxl_path = dest_att_dir / "sdxl_pilot_output.png"\n                dest_sdxl_path.write_bytes(zf.read(target_sdxl_member))\n\n            resolved_inputs[att_id] = {\n                "authentic_path": dest_auth_path,\n                "authentic_sha256": obs_auth_sha,\n                "sdxl_path": dest_sdxl_path,\n            }\n            print(f"[PASS] Extracted and verified authentic input for {att_id}: {obs_auth_sha[:16]}...")\n\n    return resolved_inputs\n\n\ndef resolve_inputs_from_directory(\n    inputs_dir: Path,\n    plan: Dict[str, Any],\n) -> Dict[str, Dict[str, Any]]:\n    """Resolves authentic inputs from an existing directory (fallback)."""\n    resolved_inputs: Dict[str, Dict[str, Any]] = {}\n    attempts = plan["attempts"]\n\n    for att in attempts:\n        att_id = att["attempt_id"]\n        row_idx = att.get("source_row_index")\n        expected_sha = att["authentic_sha256"]\n\n        candidates = [\n            inputs_dir / att_id / "authentic_control.png",\n            inputs_dir / f"{att_id}.png",\n            inputs_dir / ("attempt_0_replacement_price" if row_idx == 22 else "attempt_1_replacement_item_name") / "authentic_control.png",\n        ]\n\n        found_path: Optional[Path] = None\n        for c in candidates:\n            if c.is_file():\n                found_path = c\n                break\n\n        if not found_path:\n            # Glob search\n            for p in inputs_dir.glob("**/authentic_control.png"):\n                if compute_sha256(p) == expected_sha:\n                    found_path = p\n                    break\n\n        if not found_path:\n            raise FileNotFoundError(f"Could not locate authentic control for {att_id} in {inputs_dir}")\n\n        obs_sha = compute_sha256(found_path)\n        if obs_sha != expected_sha:\n            raise ValueError(f"Authentic SHA mismatch for {att_id}: expected {expected_sha}, got {obs_sha}")\n\n        sdxl_candidates = [\n            found_path.parent / "final_edited_document.png",\n            found_path.parent / "sdxl_pilot_output.png",\n        ]\n        sdxl_path = next((p for p in sdxl_candidates if p.is_file()), None)\n\n        resolved_inputs[att_id] = {\n            "authentic_path": found_path,\n            "authentic_sha256": obs_sha,\n            "sdxl_path": sdxl_path,\n        }\n        print(f"[PASS] Verified input {att_id}: {obs_sha[:16]}...")\n\n    return resolved_inputs\n\n\ndef acquire_runner_lock(lock_file: Path) -> None:\n    """Acquires a single-instance lock file with PID checking to prevent concurrent execution."""\n    if lock_file.is_file():\n        try:\n            lock_data = json.loads(lock_file.read_text(encoding="utf-8"))\n            pid = lock_data.get("pid")\n            is_alive = False\n            if pid and isinstance(pid, int):\n                try:\n                    os.kill(pid, 0)\n                    is_alive = True\n                except (OSError, ProcessLookupError, PermissionError):\n                    is_alive = False\n            if is_alive:\n                raise RuntimeError(\n                    f"Another AnyText runner process (PID {pid}) is currently running! "\n                    f"Concurrent executions are prohibited to preserve experimental integrity."\n                )\n        except RuntimeError:\n            raise\n        except Exception:\n            pass\n    lock_file.write_text(\n        json.dumps({"pid": os.getpid(), "timestamp": datetime.datetime.now().isoformat()}),\n        encoding="utf-8",\n    )\n\n\ndef release_runner_lock(lock_file: Path) -> None:\n    """Releases the lock file safely."""\n    try:\n        if lock_file.is_file():\n            lock_file.unlink(missing_ok=True)\n    except Exception:\n        pass\n\n\n# Mandatory CLIP files required for AnyText inference (clip-vit-large-patch14).\n# A non-empty CLIP directory is NOT proof that weights are complete.\nCLIP_MANDATORY_FILES = [\n    "config.json",\n    "vocab.json",\n    "tokenizer_config.json",\n]\n# At least one of these weight files must exist:\nCLIP_WEIGHT_FILE_CANDIDATES = [\n    "pytorch_model.bin",\n    "model.safetensors",\n]\n\n\ndef verify_clip_weights_complete(clip_dir: Path) -> Tuple[bool, List[str]]:\n    """Verifies that all mandatory CLIP inference files are present.\n\n    Returns:\n        Tuple of (is_complete, list_of_missing_files).\n    """\n    if not clip_dir.is_dir():\n        return False, ["directory_not_found"]\n\n    missing: List[str] = []\n    for f in CLIP_MANDATORY_FILES:\n        if not (clip_dir / f).is_file():\n            missing.append(f)\n\n    has_weights = any((clip_dir / w).is_file() for w in CLIP_WEIGHT_FILE_CANDIDATES)\n    if not has_weights:\n        missing.append(f"one_of({\', \'.join(CLIP_WEIGHT_FILE_CANDIDATES)})")\n\n    return len(missing) == 0, missing\n\n\ndef find_prior_completed_attempt(\n    output_root: Path,\n    attempt_id: str,\n    exclude_dir: Optional[Path] = None,\n) -> Optional[Path]:\n    """Finds a previously completed attempt with TECHNICAL_QC_PASS and verified outside equality."""\n    if not output_root.is_dir():\n        return None\n    for receipt_path in output_root.glob(f"*/{attempt_id}/attempt_receipt.json"):\n        try:\n            att_folder = receipt_path.parent\n            if exclude_dir and (\n                att_folder.resolve() == exclude_dir.resolve()\n                or exclude_dir.resolve() in att_folder.resolve().parents\n            ):\n                continue\n            rcpt = json.loads(receipt_path.read_text(encoding="utf-8"))\n            if (\n                rcpt.get("status") == "TECHNICAL_QC_PASS"\n                and rcpt.get("outside_mask_bitwise_equal") is True\n                and (\n                    (att_folder / "anytext_final_edited.png").is_file()\n                    or (att_folder / "final_edited_document.png").is_file()\n                )\n                and (att_folder / "anytext_raw_canvas.png").is_file()\n            ):\n                return att_folder\n        except Exception:\n            continue\n    return None\n\n\n\ndef download_and_verify_modelscope_anytext(\n    plan: Dict[str, Any],\n    cache_dir: Path,\n) -> Tuple[str, Dict[str, Any]]:\n    """Downloads AnyText weights via ModelScope snapshot_download ignoring translator and README.\n\n    Verifies checkpoint SHA-256 and records cache inventory metrics. Reuses valid cache if present.\n    """\n    model_spec = plan["model_specification"]\n    repo_id = model_spec["modelscope_repo_id"]\n    revision = model_spec["model_revision"]\n    checkpoint_file = model_spec["checkpoint_file"]\n    expected_ckpt_sha = model_spec["checkpoint_sha256"]\n    planned_bytes = model_spec["total_required_weights_bytes"]\n\n    # Read download ceiling from resource budget\n    resource_budget = plan.get("resource_budget", {})\n    max_download_gib = float(resource_budget.get("max_colab_download_gib", 12.0))\n    download_ceiling_bytes = int(max_download_gib * (1024**3))\n\n    planned_required_bytes = model_spec.get("total_required_weights_bytes", planned_bytes)\n    planned_required_gib = round(planned_required_bytes / (1024**3), 4)\n\n    # 1. Pre-download inventory verification against locked budget ceiling\n    print(f"[INFO] Pre-download inventory verification: planned required weights {planned_required_gib} GiB ({planned_required_bytes} bytes).")\n    print(f"[INFO] Locked budget ceiling: {max_download_gib} GiB ({download_ceiling_bytes} bytes).")\n\n    if planned_required_bytes > download_ceiling_bytes:\n        raise RuntimeError(\n            f"Pre-download inventory check failed: planned required weights ({planned_required_gib} GiB) "\n            f"exceed locked budget ceiling ({max_download_gib} GiB)."\n        )\n\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    os.environ["MODELSCOPE_CACHE"] = str(cache_dir.resolve())\n\n    # Pre-inventory of cache_dir before network download\n    pre_inventory: Dict[str, int] = {}\n    for f in cache_dir.rglob("*"):\n        if f.is_file():\n            try:\n                pre_inventory[str(f.resolve())] = f.stat().st_size\n            except OSError:\n                pass\n    pre_existing_cache_bytes = sum(pre_inventory.values())\n    print(f"[INFO] Pre-download cache inventory: {len(pre_inventory)} files, {pre_existing_cache_bytes / (1024**3):.4f} GiB already present.")\n\n    # 2. Check if valid checkpoint and CLIP weights are already present in cache\n    existing_ckpt_path: Optional[Path] = None\n    for p in cache_dir.rglob(checkpoint_file):\n        if p.is_file():\n            existing_ckpt_path = p\n            break\n\n    cache_reused = False\n    download_duration = 0.0\n    model_dir_path: Optional[Path] = None\n\n    if existing_ckpt_path and existing_ckpt_path.is_file():\n        print(f"[INFO] Inspecting existing checkpoint in cache: {existing_ckpt_path}")\n        try:\n            obs_sha = compute_sha256(existing_ckpt_path)\n            if obs_sha == expected_ckpt_sha:\n                clip_sub = existing_ckpt_path.parent / "clip-vit-large-patch14"\n                clip_ok, clip_missing = verify_clip_weights_complete(clip_sub)\n                if clip_ok:\n                    print("[PASS] Verified existing checkpoint SHA-256 and CLIP weights in cache! Reusing valid cache.")\n                    model_dir_path = existing_ckpt_path.parent\n                    cache_reused = True\n                else:\n                    print(f"[INFO] Checkpoint matched but CLIP weights incomplete in cache (missing: {clip_missing}). Will proceed with download.")\n            else:\n                print(f"[INFO] Checkpoint SHA mismatch in cache ({obs_sha[:8]} != {expected_ckpt_sha[:8]}). Will re-download.")\n        except Exception as e:\n            print(f"[INFO] Cache inspection check error: {e}")\n\n    if not cache_reused:\n        from modelscope.hub.snapshot_download import snapshot_download\n\n        print(f"[INFO] Initiating ModelScope snapshot download for {repo_id} ({revision})...")\n        print(f"[INFO] Excluding \'nlp_csanmt_translation_zh2en/*\', \'README.md\', and \'*.md\' to preserve budget and avoid download metadata errors...")\n\n        t0 = time.time()\n        try:\n            model_dir = snapshot_download(\n                repo_id,\n                revision=revision,\n                cache_dir=str(cache_dir.resolve()),\n                ignore_file_pattern=["nlp_csanmt_translation_zh2en/*", "README.md", "*.md"],\n            )\n            download_duration = time.time() - t0\n            model_dir_path = Path(model_dir)\n        except Exception as dl_err:\n            download_duration = time.time() - t0\n            # If download threw an error (e.g. README.md or transient metadata error), check if checkpoint exists and matches SHA\n            found_ckpt: Optional[Path] = None\n            for p in cache_dir.rglob(checkpoint_file):\n                if p.is_file():\n                    found_ckpt = p\n                    break\n            if found_ckpt and compute_sha256(found_ckpt) == expected_ckpt_sha:\n                # Checkpoint is intact — also verify CLIP weights are complete\n                clip_sub_err = found_ckpt.parent / "clip-vit-large-patch14"\n                clip_ok_err, clip_missing_err = verify_clip_weights_complete(clip_sub_err)\n                if not clip_ok_err:\n                    failure_receipt = {\n                        "status": "FAILED",\n                        "stage": "snapshot_download_error_handler",\n                        "error": str(dl_err),\n                        "checkpoint_sha256_matched": True,\n                        "clip_complete": False,\n                        "clip_missing_files": clip_missing_err,\n                        "message": "Checkpoint intact but CLIP weights incomplete after download error. Cannot continue.",\n                    }\n                    receipt_path = found_ckpt.parent / "failure_receipt.json"\n                    receipt_path.write_text(json.dumps(failure_receipt, indent=2), encoding="utf-8")\n                    print(f"[FAIL] Checkpoint intact but CLIP incomplete (missing: {clip_missing_err}). Failure receipt: {receipt_path}")\n                    raise RuntimeError(\n                        f"CLIP weights incomplete after snapshot_download error (missing: {clip_missing_err}). "\n                        f"Failure receipt written to {receipt_path}"\n                    ) from dl_err\n                print(f"[WARN] snapshot_download encountered non-fatal error: {dl_err}, but required checkpoint and CLIP weights are intact. Reusing cache.")\n                model_dir_path = found_ckpt.parent\n            else:\n                raise dl_err\n\n    assert model_dir_path is not None, "Model directory path resolution failed."\n\n    # Check translator directory was excluded\n    translator_dir = model_dir_path / "nlp_csanmt_translation_zh2en"\n    has_translator = translator_dir.is_dir() and any(translator_dir.iterdir())\n    if has_translator:\n        raise RuntimeError("Translator directory was unexpectedly downloaded! Exceeds budget constraints.")\n\n    # Post-inventory of cache_dir to measure on-disk cache growth\n    post_inventory: Dict[str, int] = {}\n    for f in cache_dir.rglob("*"):\n        if f.is_file():\n            try:\n                post_inventory[str(f.resolve())] = f.stat().st_size\n            except OSError:\n                pass\n\n    cache_total_files_bytes = sum(f.stat().st_size for f in model_dir_path.rglob("*") if f.is_file())\n    cache_total_files_gib = round(cache_total_files_bytes / (1024**3), 4)\n\n    # Observed cache growth on disk during snapshot_download (filesystem delta, not direct network traffic)\n    growth_bytes = 0\n    for p, sz in post_inventory.items():\n        if p not in pre_inventory:\n            growth_bytes += sz\n        elif sz > pre_inventory[p]:\n            growth_bytes += (sz - pre_inventory[p])\n    observed_cache_growth_bytes = growth_bytes\n    observed_cache_growth_gib = round(observed_cache_growth_bytes / (1024**3), 4)\n\n    # Supplementary post-download check\n    if observed_cache_growth_bytes > download_ceiling_bytes:\n        raise RuntimeError(\n            f"Supplementary post-download check failed: observed cache growth {observed_cache_growth_gib} GiB "\n            f"exceeded budget ceiling {max_download_gib} GiB."\n        )\n\n    # Checkpoint hash verification\n    ckpt_path = model_dir_path / checkpoint_file\n    if not ckpt_path.is_file():\n        raise FileNotFoundError(f"Checkpoint file not found: {ckpt_path}")\n\n    observed_ckpt_sha = compute_sha256(ckpt_path)\n    if observed_ckpt_sha != expected_ckpt_sha:\n        raise ValueError(\n            f"Checkpoint SHA-256 mismatch! Expected {expected_ckpt_sha}, got {observed_ckpt_sha}"\n        )\n\n    # CLIP weights completeness verification (fail-closed)\n    clip_dir_post = model_dir_path / "clip-vit-large-patch14"\n    clip_complete_post, clip_missing_post = verify_clip_weights_complete(clip_dir_post)\n    if not clip_complete_post:\n        failure_receipt = {\n            "status": "FAILED",\n            "stage": "post_download_clip_verification",\n            "checkpoint_sha256_matched": True,\n            "clip_complete": False,\n            "clip_missing_files": clip_missing_post,\n            "message": "Checkpoint verified but CLIP weights incomplete. Cannot proceed to inference.",\n        }\n        failure_path = model_dir_path / "failure_receipt.json"\n        failure_path.write_text(json.dumps(failure_receipt, indent=2), encoding="utf-8")\n        raise RuntimeError(\n            f"CLIP weights incomplete after download (missing: {clip_missing_post}). "\n            f"Failure receipt: {failure_path}"\n        )\n\n    receipt = {\n        "model_dir": str(model_dir_path),\n        "repo_id": repo_id,\n        "revision": revision,\n        "download_duration_sec": round(download_duration, 2),\n        "planned_weights_bytes": planned_bytes,\n        "planned_weights_gib": planned_required_gib,\n        "download_ceiling_gib": max_download_gib,\n        "download_ceiling_bytes": download_ceiling_bytes,\n        "pre_download_inventory_verified": True,\n        "cache_reused": cache_reused,\n        "cache_pre_existing_bytes": pre_existing_cache_bytes,\n        "cache_total_files_bytes": cache_total_files_bytes,\n        "cache_total_files_gib": cache_total_files_gib,\n        "observed_cache_growth_bytes": observed_cache_growth_bytes,\n        "observed_cache_growth_gib": observed_cache_growth_gib,\n        "network_traffic_bytes": "NOT_MEASURED",\n        "network_accounting_status": "NOT_MEASURED",\n        "network_accounting_description": "Cache growth observed via pre/post directory inventory diff, not direct socket network measurement.",\n        "within_budget_ceiling": observed_cache_growth_bytes <= download_ceiling_bytes,\n        "translator_excluded": True,\n        "readme_excluded": True,\n        "checkpoint_sha256": observed_ckpt_sha,\n        "checkpoint_status": "VERIFIED_MATCH",\n        "clip_complete": True,\n        "clip_missing_files": [],\n    }\n    print(f"[PASS] Model cache verified: {cache_total_files_gib} GiB on disk, observed cache growth: {observed_cache_growth_gib} GiB (within {max_download_gib} GiB ceiling).")\n    print(f"[PASS] Network traffic note: Direct network socket traffic NOT_MEASURED (cache growth is reported as on-disk filesystem delta).")\n    print(f"[PASS] Checkpoint verified: {observed_ckpt_sha[:16]}...")\n    print(f"[PASS] CLIP weights verified: all mandatory files present.")\n    return str(model_dir_path), receipt\n\n\ndef run_anytext_inference(\n    pipe: Any,\n    ori_image_canvas: Image.Image,\n    draw_pos_canvas: Image.Image,\n    prompt: str,\n    target_text: str,\n    seed: int,\n    sampling_params: Dict[str, Any],\n    a_prompt: str,\n    n_prompt: str,\n) -> Tuple[Image.Image, float, Dict[str, Any]]:\n    """Invokes AnyText pipeline in text-editing mode with CUDA synchronization."""\n    import torch\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n\n    t0 = time.time()\n\n    ori_np = np.array(ori_image_canvas.convert("RGB"))\n    pos_np = np.array(draw_pos_canvas.convert("L"))[..., np.newaxis]\n\n    input_data = {\n        "prompt": prompt,\n        "seed": seed,\n        "draw_pos": pos_np,\n        "ori_image": ori_np,\n    }\n\n    forward_params = {\n        "mode": "text-editing",\n        "image_count": 1,\n        "ddim_steps": sampling_params.get("ddim_steps", 20),\n        "cfg_scale": sampling_params.get("cfg_scale", 9.0),\n        "strength": sampling_params.get("strength", 1.0),\n        "eta": sampling_params.get("eta", 0.0),\n        "a_prompt": a_prompt,\n        "n_prompt": n_prompt,\n        "show_debug": False,\n        "revise_pos": False,\n    }\n\n    results, rtn_code, rtn_warning, debug_info = pipe(input_data, **forward_params)\n\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n    latency_sec = time.time() - t0\n\n    if rtn_code < 0 or not results:\n        raise RuntimeError(f"AnyText inference failed with code {rtn_code}: {rtn_warning}")\n\n    raw_output_np = results[0]\n    raw_output_img = Image.fromarray(raw_output_np.astype(np.uint8), mode="RGB")\n\n    peak_vram_mb = (\n        round(torch.cuda.max_memory_allocated() / (1024**2), 2)\n        if torch.cuda.is_available()\n        else "NOT_MEASURED"\n    )\n\n    meta = {\n        "latency_sec": round(latency_sec, 2),\n        "rtn_code": rtn_code,\n        "rtn_warning": rtn_warning,\n        "peak_vram_mb": peak_vram_mb,\n    }\n    return raw_output_img, latency_sec, meta\n\n\ndef crop_zoomed_roi(\n    image: Image.Image,\n    target_bbox: List[int],\n    margin_ratio_y: float = 0.8,\n    margin_ratio_x: float = 0.4,\n    min_pad_y: int = 30,\n    min_pad_x: int = 60,\n) -> Image.Image:\n    """Crops a zoomed region of interest around target_bbox [ymin, xmin, ymax, xmax]."""\n    doc_w, doc_h = image.size\n    ymin, xmin, ymax, xmax = target_bbox\n\n    h_box = ymax - ymin\n    w_box = xmax - xmin\n\n    pad_y = max(min_pad_y, int(h_box * margin_ratio_y))\n    pad_x = max(min_pad_x, int(w_box * margin_ratio_x))\n\n    crop_ymin = max(0, ymin - pad_y)\n    crop_ymax = min(doc_h, ymax + pad_y)\n    crop_xmin = max(0, xmin - pad_x)\n    crop_xmax = min(doc_w, xmax + pad_x)\n\n    return image.crop((crop_xmin, crop_ymin, crop_xmax, crop_ymax))\n\n\ndef generate_anytext_contact_sheet(\n    attempts_data: List[Dict[str, Any]],\n    output_path: Path,\n) -> None:\n    """Generates an enhanced comparison contact sheet with zoomed ROIs."""\n    cell_w, cell_h = 360, 360\n    header_h = 75\n    label_h = 35\n    cols = 6  # 1. Auth ROI, 2. SDXL ROI, 3. AnyText Raw ROI, 4. AnyText Final ROI, 5. AnyText Final Full, 6. Mask ROI\n    rows = len(attempts_data)\n    margin = 15\n\n    total_w = cols * cell_w + (cols + 1) * margin\n    total_h = header_h + rows * (cell_h + label_h) + (rows + 1) * margin\n\n    sheet = Image.new("RGB", (total_w, total_h), (250, 250, 252))\n    draw = ImageDraw.Draw(sheet)\n\n    # Header\n    draw.text(\n        (margin, 12),\n        "AnyText Document Feasibility Study: Zoomed ROI Comparison (Attempt A & B)",\n        fill=(15, 23, 42),\n    )\n    draw.text(\n        (margin, 38),\n        "Descriptive comparison between SDXL (1024x1024) and AnyText (512x512 canvas). Not an isolated architecture study.",\n        fill=(71, 85, 105),\n    )\n\n    col_titles = [\n        "1. Authentic ROI (Zoomed)",\n        "2. SDXL Pilot ROI (Zoomed)",\n        "3. AnyText Raw Canvas ROI",\n        "4. AnyText Final ROI (Zoomed)",\n        "5. AnyText Final Full Document",\n        "6. Intervention Mask (Zoomed)",\n    ]\n    for c_idx, title in enumerate(col_titles):\n        x = margin + c_idx * (cell_w + margin)\n        draw.text((x, 56), title, fill=(30, 41, 59))\n\n    for r_idx, att in enumerate(attempts_data):\n        y_top = header_h + margin + r_idx * (cell_h + label_h + margin)\n        draw.text(\n            (margin, y_top),\n            f"Attempt {att[\'id\']} | Target: \'{att[\'target_text\']}\' | Seed: {att[\'seed\']} | Latency: {att.get(\'latency\', \'N/A\')}s",\n            fill=(2, 132, 199),\n        )\n\n        cells = [\n            att.get("authentic_roi"),\n            att.get("sdxl_roi"),\n            att.get("anytext_raw_roi"),\n            att.get("anytext_final_roi"),\n            att.get("anytext_final_full"),\n            att.get("mask_roi"),\n        ]\n\n        for c_idx, img in enumerate(cells):\n            x = margin + c_idx * (cell_w + margin)\n            y = y_top + label_h\n            if img:\n                thumb = img.copy()\n                thumb.thumbnail((cell_w, cell_h), Image.Resampling.BICUBIC)\n                offset_x = x + (cell_w - thumb.width) // 2\n                offset_y = y + (cell_h - thumb.height) // 2\n                sheet.paste(thumb, (offset_x, offset_y))\n                draw.rectangle([offset_x, offset_y, offset_x + thumb.width, offset_y + thumb.height], outline=(203, 213, 225), width=1)\n            else:\n                draw.rectangle([x, y, x + cell_w, y + cell_h], fill=(241, 245, 249))\n                draw.text((x + 20, y + cell_h // 2 - 10), "N/A (Not Provided)", fill=(148, 163, 184))\n\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(output_path, "PNG")\n    print(f"[OK] Enhanced contact sheet written to: {output_path}")\n\n\ndef package_results_zip(\n    run_dir: Path,\n    output_zip_path: Path,\n    max_size_bytes: int = 100 * 1024 * 1024,\n) -> Path:\n    """Packages run artifacts into zip enforcing strict whitelist and <= 100 MiB limit."""\n    allowed_extensions = {".png", ".json", ".csv"}\n    disallowed_keywords = {".ckpt", ".pt", ".bin", ".cache", ".py", ".pth", ".safetensors"}\n\n    if output_zip_path.exists():\n        output_zip_path.unlink()\n\n    with zipfile.ZipFile(output_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:\n        for root_p, _, files in os.walk(run_dir):\n            for file in files:\n                p = Path(root_p) / file\n                ext = p.suffix.lower()\n\n                # Filter by whitelist\n                if ext not in allowed_extensions:\n                    continue\n                if any(kw in file.lower() for kw in disallowed_keywords):\n                    continue\n\n                rel_name = p.relative_to(run_dir)\n                zf.write(p, arcname=str(Path(run_dir.name) / rel_name))\n\n    zip_size = output_zip_path.stat().st_size\n    if zip_size > max_size_bytes:\n        output_zip_path.unlink(missing_ok=True)\n        raise ValueError(\n            f"Result archive exceeded 100 MiB limit! ({zip_size / (1024**2):.2f} MiB). Archive deleted."\n        )\n\n    print(f"[OK] Results safely archived to: {output_zip_path}")\n    print(f"     Archive Size: {zip_size / (1024**2):.2f} MiB (Ceiling: 100 MiB)")\n    print(f"     SHA-256: {compute_sha256(output_zip_path)}")\n    return output_zip_path\n\n\ndef main() -> int:\n    # Ensure script directory and current working directory are in sys.path\n    script_dir = Path(__file__).resolve().parent\n    if str(script_dir) not in sys.path:\n        sys.path.insert(0, str(script_dir))\n    if str(Path.cwd()) not in sys.path:\n        sys.path.insert(0, str(Path.cwd()))\n\n    parser = argparse.ArgumentParser(description="Run AnyText Feasibility Runner")\n    parser.add_argument("--plan-path", type=str, required=True, help="Path to anytext_feasibility_plan.json")\n    parser.add_argument("--inputs-zip", type=str, default=None, help="Path to document_feasibility_results.zip")\n    parser.add_argument("--inputs-dir", type=str, default=None, help="Path to inputs directory")\n    parser.add_argument("--output-root", type=str, required=True, help="Path to outputs root directory")\n    parser.add_argument("--cache-dir", type=str, default=None, help="Path to ModelScope cache directory")\n    parser.add_argument("--custom-font", type=str, default=None, help="Custom TTF font path")\n    parser.add_argument("--preflight-only", action="store_true", help="Run preflight checks only")\n    args = parser.parse_args()\n\n    plan_path = Path(args.plan_path).resolve()\n    output_root = Path(args.output_root).resolve()\n    output_root.mkdir(parents=True, exist_ok=True)\n\n    lock_file = output_root / ".runner.lock"\n    try:\n        acquire_runner_lock(lock_file)\n    except RuntimeError as lock_err:\n        print(f"[FAIL] {lock_err}")\n        return 1\n\n    current_stage = "INITIALIZATION"\n    run_dir: Optional[Path] = None\n    plan_bytes_sha = "UNKNOWN"\n\n    try:\n        if not plan_path.is_file():\n            raise FileNotFoundError(f"Plan not found: {plan_path}")\n\n        plan = json.loads(plan_path.read_text(encoding="utf-8"))\n        plan_bytes_sha = hashlib.sha256(plan_path.read_bytes()).hexdigest()\n\n        run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")\n        run_id = f"run_anytext_{run_timestamp}_{plan_bytes_sha[:8]}"\n        run_dir = output_root / run_id\n        run_dir.mkdir(parents=True, exist_ok=True)\n\n        # Cache directory isolation: strictly outside run_dir and output_root\n        if args.cache_dir:\n            model_cache_dir = Path(args.cache_dir).resolve()\n        else:\n            # Default Colab location /content/modelscope_cache, or sibling on local\n            model_cache_dir = Path("/content/modelscope_cache") if Path("/content").is_dir() else output_root.parent / ".cache_modelscope"\n\n        print(f"=== ANYTEXT FEASIBILITY RUNNER: {run_id} ===")\n        print(f"Plan Path: {plan_path} (SHA-256: {plan_bytes_sha[:16]}...)")\n        print(f"Model Cache Dir: {model_cache_dir} (ISOLATED)")\n\n        current_stage = "INPUT_EXTRACTION"\n        print("\\n--- Resolving Authentic Inputs ---")\n        extracted_inputs: Dict[str, Dict[str, Any]] = {}\n        if args.inputs_zip and Path(args.inputs_zip).is_file():\n            staging_dir = run_dir / ".inputs_staging"\n            extracted_inputs = safe_extract_archive_inputs(\n                archive_path=Path(args.inputs_zip).resolve(),\n                target_extract_dir=staging_dir,\n                plan=plan,\n            )\n        elif args.inputs_dir and Path(args.inputs_dir).is_dir():\n            extracted_inputs = resolve_inputs_from_directory(\n                inputs_dir=Path(args.inputs_dir).resolve(),\n                plan=plan,\n            )\n        else:\n            # Check default archive path from plan\n            default_archive = Path(plan.get("input_source_specification", {}).get("archive_path", ""))\n            if not default_archive.is_file():\n                default_archive = Path(__file__).resolve().parent.parent.parent / default_archive\n            if default_archive.is_file():\n                staging_dir = run_dir / ".inputs_staging"\n                extracted_inputs = safe_extract_archive_inputs(\n                    archive_path=default_archive,\n                    target_extract_dir=staging_dir,\n                    plan=plan,\n                )\n            else:\n                raise FileNotFoundError("Neither valid --inputs-zip nor --inputs-dir provided, and default archive not found.")\n\n        current_stage = "PREFLIGHT"\n        hw_info = check_gpu_and_resources()\n        print(f"CUDA Available: {hw_info[\'cuda_available\']} ({hw_info[\'device_name\']}, VRAM: {hw_info[\'vram_total_gb\']} GB)")\n        print(f"Disk Free: {hw_info[\'disk_free_gb\']} GB")\n\n        font_path, font_sha256 = resolve_font_path(args.custom_font)\n        print(f"Resolved Font: {font_path} (SHA-256: {font_sha256[:16]}...)")\n\n        preflight_receipt = {\n            "run_id": run_id,\n            "timestamp": datetime.datetime.now().isoformat(),\n            "plan_sha256": plan_bytes_sha,\n            "hardware": hw_info,\n            "font": {"path": font_path, "sha256": font_sha256},\n            "inputs_verified": {k: v["authentic_sha256"] for k, v in extracted_inputs.items()},\n            "cache_dir": str(model_cache_dir),\n            "status": "PREFLIGHT_PASS",\n        }\n        (run_dir / "preflight_receipt.json").write_text(json.dumps(preflight_receipt, indent=2), encoding="utf-8")\n\n        if args.preflight_only:\n            print("[OK] Preflight completed successfully.")\n            return 0\n\n        if not hw_info["cuda_available"]:\n            raise RuntimeError("CUDA GPU is required for AnyText generation. Running on CPU is not supported.")\n\n        current_stage = "MODEL_DOWNLOAD"\n        print("\\n--- Downloading AnyText Model (Excluding Translator & Docs) ---")\n        model_dir, model_receipt = download_and_verify_modelscope_anytext(\n            plan=plan,\n            cache_dir=model_cache_dir,\n        )\n        (run_dir / "model_download_receipt.json").write_text(json.dumps(model_receipt, indent=2), encoding="utf-8")\n\n        current_stage = "PIPELINE_INIT"\n        print("\\n--- Initializing AnyText Pipeline from Local Weights ---")\n        from modelscope.pipelines import pipeline\n\n        pipe = pipeline(\n            "my-anytext-task",\n            model=model_dir,\n            use_fp16=True,\n            use_translator=False,\n            font_path=font_path,\n        )\n        print("[OK] AnyText pipeline initialized successfully.")\n\n        # Import geometry adapter\n        from ml.evaluation.anytext_document_adapter import (\n            prepare_anytext_editing_inputs,\n            composite_anytext_editing_output,\n        )\n\n        current_stage = "INFERENCE"\n        contact_sheet_items: List[Dict[str, Any]] = []\n        review_records: List[Dict[str, Any]] = []\n        attempts = plan["attempts"]\n        has_failure = False\n\n        for att in attempts:\n            att_id = att["attempt_id"]\n            att_dir = run_dir / att_id\n            att_dir.mkdir(parents=True, exist_ok=True)\n            print(f"\\n--- Checking / Executing {att_id} ---")\n\n            # Check if prior completed attempt exists\n            prior_att_folder = find_prior_completed_attempt(output_root, att_id, exclude_dir=run_dir)\n            if prior_att_folder:\n                print(f"[PASS] Prior completed attempt found at: {prior_att_folder}")\n                # Copy artifacts to att_dir if different\n                if prior_att_folder.resolve() != att_dir.resolve():\n                    for src_item in prior_att_folder.iterdir():\n                        if src_item.is_file():\n                            shutil.copy2(src_item, att_dir / src_item.name)\n\n                prior_rcpt_file = att_dir / "attempt_receipt.json"\n                if prior_rcpt_file.is_file():\n                    att_receipt = json.loads(prior_rcpt_file.read_text(encoding="utf-8"))\n                    print(f"[INFO] Reusing verified result for {att_id} (outside_mask_bitwise_equal={att_receipt.get(\'outside_mask_bitwise_equal\')}).")\n\n                    auth_roi_path = att_dir / "authentic_roi_zoomed.png"\n                    sdxl_roi_path = att_dir / "sdxl_roi_zoomed.png"\n                    raw_roi_path = att_dir / "anytext_raw_roi_zoomed.png"\n                    final_roi_path = att_dir / "anytext_final_roi_zoomed.png"\n                    final_img_path = att_dir / "anytext_final_edited.png"\n                    if not final_img_path.is_file():\n                        final_img_path = att_dir / "final_edited_document.png"\n                    mask_roi_path = att_dir / "mask_roi_zoomed.png"\n\n                    contact_sheet_items.append({\n                        "id": att_id,\n                        "target_text": att["target_text"],\n                        "seed": att["seed"],\n                        "latency": att_receipt.get("latency_sec", 0.0),\n                        "authentic_roi": Image.open(auth_roi_path).convert("RGB") if auth_roi_path.is_file() else None,\n                        "sdxl_roi": Image.open(sdxl_roi_path).convert("RGB") if sdxl_roi_path.is_file() else None,\n                        "anytext_raw_roi": Image.open(raw_roi_path).convert("RGB") if raw_roi_path.is_file() else None,\n                        "anytext_final_roi": Image.open(final_roi_path).convert("RGB") if final_roi_path.is_file() else None,\n                        "anytext_final_full": Image.open(final_img_path).convert("RGB") if final_img_path.is_file() else None,\n                        "mask_roi": Image.open(mask_roi_path).convert("RGB") if mask_roi_path.is_file() else None,\n                    })\n\n                    review_records.append({\n                        "attempt_id": att_id,\n                        "source_row_index": att["source_row_index"],\n                        "original_text": att["original_text"],\n                        "target_text": att["target_text"],\n                        "exact_text_success": "PENDING_HUMAN_REVIEW",\n                        "visual_realism": "PENDING_HUMAN_REVIEW",\n                        "human_verdict": "PENDING_HUMAN_REVIEW",\n                        "notes": "Reused prior verified attempt; awaiting human reviewer assessment.",\n                    })\n                    continue\n\n            auth_info = extracted_inputs.get(att_id)\n            if not auth_info:\n                print(f"[FAIL] Missing authentic input data for {att_id}")\n                has_failure = True\n                fail_receipt = {\n                    "attempt_id": att_id,\n                    "status": "FAILED",\n                    "error": f"Missing authentic input data for {att_id}",\n                    "traceback": "",\n                }\n                (att_dir / "failure_receipt.json").write_text(json.dumps(fail_receipt, indent=2), encoding="utf-8")\n                continue\n\n            auth_img = Image.open(auth_info["authentic_path"]).convert("RGB")\n            sdxl_img = Image.open(auth_info["sdxl_path"]).convert("RGB") if auth_info.get("sdxl_path") and auth_info["sdxl_path"].is_file() else None\n\n            try:\n                # 1. Geometry preparation\n                prep_res = prepare_anytext_editing_inputs(\n                    image=auth_img,\n                    target_bbox=att["target_bbox"],\n                    canvas_size=att["sampling_parameters"].get("canvas_width", 512),\n                    context_margin_ratio=att["sampling_parameters"].get("context_margin_ratio", 0.5),\n                )\n\n                # 2. Run AnyText inference\n                raw_canvas, latency, inf_meta = run_anytext_inference(\n                    pipe=pipe,\n                    ori_image_canvas=prep_res["ori_image_canvas"],\n                    draw_pos_canvas=prep_res["draw_pos_canvas"],\n                    prompt=att["prompt"],\n                    target_text=att["target_text"],\n                    seed=att["seed"],\n                    sampling_params=att["sampling_parameters"],\n                    a_prompt=att["a_prompt"],\n                    n_prompt=att["n_prompt"],\n                )\n\n                # 3. Composite output strictly inside target mask\n                comp_res = composite_anytext_editing_output(\n                    original_image=auth_img,\n                    raw_anytext_output=raw_canvas,\n                    geometry_receipt=prep_res["geometry_receipt"],\n                    native_mask=prep_res["native_mask"],\n                )\n\n                final_img = comp_res["final_edited_document"]\n                is_outside_equal = comp_res["is_outside_equal"]\n                max_outside_diff = comp_res["max_outside_diff"]\n\n                if not is_outside_equal:\n                    raise ValueError(f"Outside-mask equality contract violated! Max outside diff: {max_outside_diff}")\n                print(f"[PASS] Outside-mask equality verified (diff = 0).")\n\n                # 4. Save artifacts\n                auth_save = att_dir / "authentic_control.png"\n                raw_save = att_dir / "anytext_raw_canvas.png"\n                final_save = att_dir / "anytext_final_edited.png"\n                mask_save = att_dir / "native_mask.png"\n                draw_pos_save = att_dir / "draw_pos_canvas.png"\n\n                auth_img.save(auth_save, "PNG")\n                raw_canvas.save(raw_save, "PNG")\n                final_img.save(final_save, "PNG")\n                prep_res["native_mask"].save(mask_save, "PNG")\n                prep_res["draw_pos_canvas"].save(draw_pos_save, "PNG")\n\n                # Compute ZOOMED ROIs for contact sheet\n                target_bbox = att["target_bbox"]\n                auth_roi = crop_zoomed_roi(auth_img, target_bbox)\n                sdxl_roi = crop_zoomed_roi(sdxl_img, target_bbox) if sdxl_img else None\n                anytext_final_roi = crop_zoomed_roi(final_img, target_bbox)\n                mask_roi = crop_zoomed_roi(prep_res["native_mask"].convert("RGB"), target_bbox)\n\n                # Crop ROI from raw canvas around target box in canvas\n                t_box_canvas = prep_res["geometry_receipt"]["target_bbox_in_canvas"]\n                raw_roi = crop_zoomed_roi(raw_canvas, t_box_canvas)\n\n                # Save ROI images\n                auth_roi.save(att_dir / "authentic_roi_zoomed.png", "PNG")\n                anytext_final_roi.save(att_dir / "anytext_final_roi_zoomed.png", "PNG")\n                raw_roi.save(att_dir / "anytext_raw_roi_zoomed.png", "PNG")\n                mask_roi.save(att_dir / "mask_roi_zoomed.png", "PNG")\n                if sdxl_roi:\n                    sdxl_roi.save(att_dir / "sdxl_roi_zoomed.png", "PNG")\n\n                att_receipt = {\n                    "attempt_id": att_id,\n                    "status": "TECHNICAL_QC_PASS",\n                    "operation": att["operation"],\n                    "target_text": att["target_text"],\n                    "seed": att["seed"],\n                    "latency_sec": latency,\n                    "peak_vram_mb": inf_meta["peak_vram_mb"],\n                    "outside_mask_bitwise_equal": is_outside_equal,\n                    "max_outside_diff": max_outside_diff,\n                    "hashes": {\n                        "authentic_sha256": compute_sha256(auth_save),\n                        "raw_canvas_sha256": compute_sha256(raw_save),\n                        "final_edited_sha256": compute_sha256(final_save),\n                        "native_mask_sha256": compute_sha256(mask_save),\n                    },\n                    "geometry_receipt": prep_res["geometry_receipt"],\n                }\n                (att_dir / "attempt_receipt.json").write_text(json.dumps(att_receipt, indent=2), encoding="utf-8")\n\n                contact_sheet_items.append({\n                    "id": att_id,\n                    "target_text": att["target_text"],\n                    "seed": att["seed"],\n                    "latency": latency,\n                    "authentic_roi": auth_roi,\n                    "sdxl_roi": sdxl_roi,\n                    "anytext_raw_roi": raw_roi,\n                    "anytext_final_roi": anytext_final_roi,\n                    "anytext_final_full": final_img,\n                    "mask_roi": mask_roi,\n                })\n\n                review_records.append({\n                    "attempt_id": att_id,\n                    "source_row_index": att["source_row_index"],\n                    "original_text": att["original_text"],\n                    "target_text": att["target_text"],\n                    "exact_text_success": "PENDING_HUMAN_REVIEW",\n                    "visual_realism": "PENDING_HUMAN_REVIEW",\n                    "human_verdict": "PENDING_HUMAN_REVIEW",\n                    "notes": "Awaiting human reviewer assessment.",\n                })\n\n            except Exception as e:\n                has_failure = True\n                err_trace = traceback.format_exc()\n                print(f"[FAIL] Error in {att_id}: {e}")\n                fail_receipt = {\n                    "attempt_id": att_id,\n                    "status": "FAILED",\n                    "error": str(e),\n                    "traceback": err_trace,\n                }\n                (att_dir / "failure_receipt.json").write_text(json.dumps(fail_receipt, indent=2), encoding="utf-8")\n\n        current_stage = "PACKAGING"\n        # Generate contact sheet if any item succeeded\n        if contact_sheet_items:\n            cs_path = run_dir / "anytext_contact_sheet.png"\n            generate_anytext_contact_sheet(contact_sheet_items, cs_path)\n\n        # Write human review CSV\n        if review_records:\n            csv_path = run_dir / "human_review.csv"\n            with csv_path.open("w", newline="", encoding="utf-8") as f:\n                fieldnames = [\n                    "attempt_id",\n                    "source_row_index",\n                    "original_text",\n                    "target_text",\n                    "exact_text_success",\n                    "visual_realism",\n                    "human_verdict",\n                    "notes",\n                ]\n                writer = csv.DictWriter(f, fieldnames=fieldnames)\n                writer.writeheader()\n                writer.writerows(review_records)\n\n        if has_failure:\n            fail_rcpt = {\n                "run_id": run_id,\n                "stage": "INFERENCE",\n                "status": "PARTIAL_FAILURE",\n                "error": "One or more generation attempts failed.",\n                "traceback": "",\n                "timestamp": datetime.datetime.now().isoformat(),\n            }\n            (run_dir / "failure_receipt.json").write_text(json.dumps(fail_rcpt, indent=2), encoding="utf-8")\n            (output_root / "failure_receipt.json").write_text(json.dumps(fail_rcpt, indent=2), encoding="utf-8")\n\n            partial_zip = output_root / "anytext_feasibility_partial_results.zip"\n            package_results_zip(run_dir=run_dir, output_zip_path=partial_zip)\n            print(f"\\n[WARNING] Runner completed with partial failure. Partial archive created at: {partial_zip}")\n            return 1\n\n        # Package full results into zip archive\n        zip_path = output_root / "anytext_feasibility_results.zip"\n        package_results_zip(run_dir=run_dir, output_zip_path=zip_path)\n\n        # Clear old root failure receipt if present\n        (output_root / "failure_receipt.json").unlink(missing_ok=True)\n\n        print("\\n[DONE] AnyText Feasibility Run executed successfully.")\n        return 0\n\n    except Exception as err:\n        err_tb = traceback.format_exc()\n        print("\\n" + "=" * 80)\n        print(f"[RUNNER FAILURE] Stage: {current_stage}")\n        print(f"Error: {err}")\n        print(f"Exit Code: 1")\n        print(f"Traceback:\\n{err_tb}")\n        print("=" * 80)\n\n        fail_info = {\n            "run_id": run_id if run_dir else "UNKNOWN",\n            "stage": current_stage,\n            "status": "FAILED",\n            "error": str(err),\n            "traceback": err_tb,\n            "timestamp": datetime.datetime.now().isoformat(),\n            "plan_sha256": plan_bytes_sha,\n        }\n        root_fail_receipt = output_root / "failure_receipt.json"\n        root_fail_receipt.write_text(json.dumps(fail_info, indent=2), encoding="utf-8")\n        if run_dir and run_dir.is_dir():\n            (run_dir / "failure_receipt.json").write_text(json.dumps(fail_info, indent=2), encoding="utf-8")\n            partial_zip = output_root / "anytext_feasibility_partial_results.zip"\n            try:\n                package_results_zip(run_dir=run_dir, output_zip_path=partial_zip)\n                print(f"[INFO] Partial results archive created at: {partial_zip}")\n            except Exception as zip_e:\n                print(f"[WARN] Failed to package partial archive: {zip_e}")\n\n        return 1\n\n    finally:\n        release_runner_lock(lock_file)\n\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'.encode('utf-8'))
assert hashlib.sha256(p_runner.read_bytes()).hexdigest() == '8be60b3f07c6adb42251fcb90885a72858ad50b2648808527c142045f4fa6ce4', 'Runner SHA mismatch after staging!'

print('[PASS] Staged plan, geometry, adapter, and runner at /content/AnyText with bit-exact SHA-256 verification.')

In [ ]:
# Cell 4: Upload and Verify SDXL Feasibility Archive
# Archive contains authentic controls and SDXL pilot outputs from the closed pilot.
import hashlib, os, pathlib
from google.colab import files

archive_name = 'document_feasibility_results.zip'
archive_path = pathlib.Path(f'/content/{archive_name}')
expected_archive_sha = 'deadd9205207eaaa296a8d9eeb493ee48eeea76ee7afd210684187e4026ed684'

if not archive_path.is_file():
    print(f'Vui lòng tải lên tệp: {archive_name}')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        if fname.endswith('.zip'):
            archive_path.write_bytes(data)
            break

assert archive_path.is_file(), f'Không tìm thấy {archive_path}! Vui lòng upload archive SDXL.'
observed_sha = hashlib.sha256(archive_path.read_bytes()).hexdigest()
assert observed_sha == expected_archive_sha, f'Archive SHA mismatch! Expected {expected_archive_sha}, got {observed_sha}'
print(f'[PASS] Archive SDXL đã xác nhận: {archive_path} (SHA-256: {observed_sha[:16]}...)')

In [ ]:
# Cell 5: Execute AnyText Feasibility Runner using Dedicated Python 3.10 Environment
import json, os, pathlib, subprocess, sys

repo_dir = pathlib.Path('/content/AnyText')
os.chdir(str(repo_dir))

output_root = pathlib.Path('/content/outputs')
output_root.mkdir(parents=True, exist_ok=True)
log_path = output_root / 'runner_execution.log'

py_bin = '/content/anytext_env/bin/python'
cmd = [
    py_bin, '-u', str(repo_dir / 'run_anytext_colab.py'),
    '--plan-path', str(repo_dir / 'research/evidence/document_pilot/anytext_feasibility_plan.json'),
    '--inputs-zip', '/content/document_feasibility_results.zip',
    '--output-root', str(output_root),
    '--cache-dir', '/content/modelscope_cache',
    '--custom-font', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
]
print('Bắt đầu thực thi AnyText Runner trên GPU bằng Python 3.10 (-u unbuffered)...')
print(f'Log file: {log_path}\n')

with open(log_path, 'w', encoding='utf-8') as log_f:
    proc = subprocess.Popen(
        cmd,
        cwd=str(repo_dir),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in iter(proc.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
        log_f.write(line)
        log_f.flush()
    proc.wait()

exit_code = proc.returncode
if exit_code != 0:
    print('\n' + '=' * 80)
    print(f'[FAIL] Runner thất bại với exit code {exit_code}!')
    receipt_path = output_root / 'failure_receipt.json'
    if not receipt_path.is_file():
        sub_receipts = list(output_root.glob('*/failure_receipt.json'))
        if sub_receipts:
            receipt_path = sub_receipts[0]
    if receipt_path.is_file():
        try:
            rcpt = json.loads(receipt_path.read_text(encoding='utf-8'))
            print(f"Stage gây lỗi: {rcpt.get('stage', 'UNKNOWN')}")
            print(f"Lỗi: {rcpt.get('error', 'UNKNOWN')}")
            print(f"Đường dẫn receipt: {receipt_path}")
            print(f"Đường dẫn log: {log_path}")
            print('Traceback gốc:')
            print(rcpt.get('traceback', 'Không có traceback.'))
        except Exception as e:
            print(f'Không thể phân tích failure receipt: {e}')
    else:
        print(f'Đường dẫn log: {log_path}')
        print('Không tìm thấy failure_receipt.json. 50 dòng cuối log:')
        if log_path.is_file():
            lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
            for l in lines[-50:]:
                print(l)
    print('=' * 80)
    raise RuntimeError(f'AnyText Runner failed with exit code {exit_code}')
else:
    print('\n[PASS] Quá trình thực thi runner hoàn thành thành công.')

In [ ]:
# Cell 6: Visual Inspection & Download Whitelisted Results Archive
import glob, json, os, pathlib
from IPython.display import Image as IPImage, display
from google.colab import files

output_root = pathlib.Path('/content/outputs')

# 1. Hiển thị bảng đối chiếu phóng đại (Zoomed ROI Contact Sheet) nếu có
cs_files = sorted(list(output_root.glob('*/anytext_contact_sheet.png')))
if cs_files:
    print('Bảng đối chiếu phóng đại (Zoomed ROI Contact Sheet):')
    display(IPImage(str(cs_files[-1])))

# 2. Kiểm tra và tải về kết quả đã đóng gói (ưu tiên full results, fallback partial results, trần <= 100 MiB)
full_zip = output_root / 'anytext_feasibility_results.zip'
partial_zip = output_root / 'anytext_feasibility_partial_results.zip'

target_zip = None
if full_zip.is_file():
    target_zip = full_zip
    print(f'[OK] Tìm thấy full results archive: {full_zip.name}')
elif partial_zip.is_file():
    target_zip = partial_zip
    print(f'[WARNING] Không có full results; tìm thấy partial results archive: {partial_zip.name}')
else:
    print('[WARNING] Không tìm thấy archive kết quả (cả full lẫn partial).')

if target_zip:
    size_mib = target_zip.stat().st_size / (1024**2)
    assert size_mib <= 100.0, f'Archive size {size_mib:.2f} MiB vượt quá trần 100 MiB!'
    print(f'[PASS] Archive hợp lệ ({size_mib:.2f} MiB <= 100 MiB). Bắt đầu download: {target_zip.name}...')
    files.download(str(target_zip))
